# Paper Results Training Addendum

Purpose: summarize metrics generated by the new standalone training notebooks without changing `PAPER_RESULTS_TABLE.ipynb`.

This notebook reads only `TRAINING_*.csv` files from `outputs/metrics/` and writes a compact addendum table.

In [ ]:
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
METRICS_DIR = NOTEBOOK_DIR / "outputs" / "metrics"
training_files = sorted(METRICS_DIR.glob("TRAINING_*.csv"))
print("found", len(training_files), "training metric files")
for path in training_files:
    print("-", path.name)

In [ ]:
rows = []
for path in training_files:
    df = pd.read_csv(path)
    if df.empty:
        continue
    best = df.sort_values("roc_auc", ascending=False).iloc[0] if "roc_auc" in df.columns else df.iloc[-1]
    rows.append({
        "source_file": path.name,
        "best_epoch": int(best.get("epoch", -1)),
        "roc_auc": best.get("roc_auc", None),
        "accuracy": best.get("accuracy", None),
        "balanced_accuracy": best.get("balanced_accuracy", None),
        "loss": best.get("loss", None),
    })

addendum = pd.DataFrame(rows)
output_path = METRICS_DIR / "PAPER_training_addendum.csv"
addendum.to_csv(output_path, index=False)
print("saved:", output_path)
addendum

## How To Use In Paper Results

- Keep the main RQ1-RQ6 paper table unchanged.
- Use this addendum for training-protocol evidence: patient-level split, imbalance handling, ROC-AUC, and checkpoint selection.
- Do not compare SIIM-ISIC 2020 training metrics directly against HAM10000 RQ metrics unless the dataset, labels, and split protocol are clearly stated.